In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LassoCV, Lasso
from sklearn.model_selection import LeaveOneOut
from scipy.linalg import pinv
from statsmodels.stats.multitest import multipletests

In [2]:
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
#from sklearn.linear_model import LassoCV
from scipy.linalg import solve
from scipy.sparse import csc_matrix
from scipy.sparse.linalg import lsqr
from scipy.sparse.linalg import spsolve
from scipy.sparse.linalg import lsmr

In [3]:
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_regression
from joblib import Parallel, delayed
from scipy.stats import norm

In [4]:
import os
os.environ["OMP_NUM_THREADS"] = "16"  # 限制最多使用 16 核

In [5]:
# Load and preprocess X_data
df_x = pd.read_csv("x.ileum0317.csv", delimiter=",")
df_x = df_x.drop(columns=["Unnamed: 0"])  # Drop sample ID
X_data = df_x #.to_numpy()

# Load and preprocess y_data
df_y = pd.read_csv("y.ileum0317.csv", delimiter=",")
df_y = df_y.drop(columns=["Unnamed: 0"])  # Drop sample ID
y_data = df_y.to_numpy()

def loocv_sigma(X, y, alpha):
    """
    Estimate sigma using LOOCV (Leave-One-Out Cross-Validation).
    """
    n = X.shape[0]
    residuals = np.zeros(n)
    loo = LeaveOneOut()

    for i, (train_index, test_index) in enumerate(loo.split(X)):
        
        #print(f"LOOCV - Processing sample {i+1}/{n}")
            
        if i % 10 == 0:  # 每 10 个样本打印一次进度
            print(f"LOOCV Progress: {i}/{n}")
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]

        model = Lasso(alpha=alpha).fit(X_train, y_train)
        y_pred = model.predict(X_test)

        residuals[i] = y_test - y_pred

    return np.sqrt(np.mean(residuals**2))

In [6]:
def desparsified_lasso(X, y, alpha=None, feature_selection='variance', k_features=5000, var_threshold=0.01):
    """
    Perform de-sparsified Lasso (Lasso projection) for inference.
    """

    n, p = X.shape
    gene_names = X.columns
    X = X.to_numpy()

    # **Step 1: 标准化数据**
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)  # 标准化 X
    if feature_selection == 'variance':
        selector = VarianceThreshold(threshold=var_threshold)
        X_selected = selector.fit_transform(X_scaled)
        selected_indices = selector.get_support(indices=True)
    elif feature_selection == 'kbest':
        selector = SelectKBest(score_func=f_regression, k=min(k_features, p))
        X_selected = selector.fit_transform(X_scaled, y)
        #selected_indices = selector.get_support(indices=True)
        selected_genes = gene_names[selector.get_support()]  # 获取被选中的基因名
    else:
        raise ValueError("feature_selection must be 'variance' or 'kbest'")

    print(f"Reduced Feature Count: {X_selected.shape[1]}")
    
    # **Step 2: 训练 LASSO**
    if alpha is None:
        print("Training LassoCV to find best alpha...")
        lasso_cv = LassoCV(cv=5, alphas=np.logspace(-3, 1, 50), max_iter=5000, n_jobs=-1).fit(X_selected, y)
        alpha = lasso_cv.alpha_
    else:
        print(f"Using provided alpha: {alpha}")
        lasso_cv = LassoCV(cv=5, alphas=[alpha], max_iter=5000, n_jobs=-1).fit(X_selected, y)

    beta_lasso = lasso_cv.coef_
    
    # **Step 3: 计算 LOOCV 误差 sigma_hat**
    print("Computing Theta diagonal elements using LSMR...")
    Sigma_hat_sparse = (X_selected.T @ X_selected) / n  # Covariance matrix

    def solve_diagonal(i):
        e_i = np.zeros(Sigma_hat_sparse.shape[0])
        e_i[i] = 1
        return lsmr(Sigma_hat_sparse, e_i)[0][i]  # Only extract diagonal element

    Theta_diag = np.array(Parallel(n_jobs=-1)(delayed(solve_diagonal)(i) for i in range(Sigma_hat_sparse.shape[0])))


    print("Step 4: Computing Precision Matrix...")
    # **Step 4: 计算精度矩阵 Theta_hat**
    print("Using LSMR solver for large gene datasets...")
    Theta_rhs = lsmr(Sigma_hat_sparse, (X_selected.T @ (y - X_selected @ beta_lasso) / n))[0]  # 避免构造 `Sigma_hat`

    #print("Using sparse solver for large precision matrix...")
    #Theta_XT = spsolve(Sigma_hat_sparse, X_scaled.T)
        
    print("Step 5: Computing Desparsified Lasso Estimator...")
    # **Step 5: 计算去稀疏化 Lasso 估计值**
    #beta_hat = beta_lasso + Theta_XT @ (y - X_scaled @ beta_lasso) / n
    beta_hat = beta_lasso + Theta_rhs
    
    std_errors = np.sqrt(Theta_diag * np.var(y) / n)  # Estimate standard error
    z_scores = beta_hat / std_errors
    p_values = 2 * norm.sf(np.abs(z_scores))  # Two-tailed p-values

    # **Step 6: 计算标准误**
    #std_errors = np.sqrt(Theta_diag * sigma_hat**2 / n)

    # **Step 7: 计算 p-values**
    z_scores = beta_hat / std_errors
    p_values = 2 * norm.sf(np.abs(z_scores))  # 计算双尾 p 值

    # **Step 8: FDR 校正**
    _, q_values, _, _ = multipletests(p_values, method="fdr_bh")

    # **Step 9: 计算 95% 置信区间**
    ci_lower = beta_hat - 1.96 * std_errors
    ci_upper = beta_hat + 1.96 * std_errors

    # **Step 10: 组织结果**
    results_df = pd.DataFrame({
        "gene": selected_genes,  # 基因名（可以替换为真实基因）
        "beta": beta_hat,
        "std_error": std_errors,
        "pval": p_values,
        "qval": q_values,
        "ci.lower": ci_lower,
        "ci.upper": ci_upper
    })

    # **Step 11: 过滤显著结果**
    #results_df = results_df[(results_df["qval"] < 0.05) & (results_df["beta"] != 0)].sort_values(by="qval")
    results_df = results_df[(results_df["qval"] < 0.1) & (results_df["beta"] != 0)].sort_values(by="qval")
    
    return results_df

In [7]:
list_ileum = [] 
for i, microbe in enumerate(df_y.columns):
    y_i = y_data[:, i]  # 选择一个微生物作为响应变量
    print(f"Results for {microbe}:")
    results_ileum = desparsified_lasso(X_data, y_i, feature_selection='kbest', k_features=2000)
    print(results_ileum)
    if not results_ileum.empty:
        results_ileum["microbe"] = microbe
        list_ileum.append(results_ileum)
    else:
        print(f"⚠️ Warning: No significant results for {microbe}!")

Results for Actinomyces_graevenitzii:
Reduced Feature Count: 2000
Training LassoCV to find best alpha...
Computing Theta diagonal elements using LSMR...
Step 4: Computing Precision Matrix...
Using LSMR solver for large gene datasets...
Step 5: Computing Desparsified Lasso Estimator...
                 gene      beta  std_error          pval          qval  \
648   ENSG00000129951  1.089832   0.025947  0.000000e+00  0.000000e+00   
1448  ENSG00000182896  0.662427   0.031390  7.421837e-99  7.421837e-96   
1391  ENSG00000178115  0.817425   0.046343  1.245981e-69  8.306540e-67   
1751  ENSG00000226363  0.669824   0.038974  3.363017e-66  1.681509e-63   
305   ENSG00000105374  0.349055   0.027559  9.172452e-37  3.668981e-34   
1802  ENSG00000236366  0.834623   0.067365  2.977303e-35  9.924344e-33   
826   ENSG00000139324 -0.281528   0.024787  6.769704e-30  1.934201e-27   
1865  ENSG00000253455  0.217341   0.024320  4.012619e-19  1.003155e-16   
1289  ENSG00000171495  0.159381   0.018993  4.79

Computing Theta diagonal elements using LSMR...
Step 4: Computing Precision Matrix...
Using LSMR solver for large gene datasets...
Step 5: Computing Desparsified Lasso Estimator...
                gene      beta  std_error  pval  qval  ci.lower  ci.upper
667  ENSG00000138449 -0.560406   0.013783   0.0   0.0 -0.587421  -0.53339
Results for Bacteroides_uniformis:
Reduced Feature Count: 2000
Training LassoCV to find best alpha...
Computing Theta diagonal elements using LSMR...
Step 4: Computing Precision Matrix...
Using LSMR solver for large gene datasets...
Step 5: Computing Desparsified Lasso Estimator...
Empty DataFrame
Columns: [gene, beta, std_error, pval, qval, ci.lower, ci.upper]
Index: []
⚠️ Warning: No significant results for Bacteroides_uniformis!
Results for Bacteroides_xylanisolvens:
Reduced Feature Count: 2000
Training LassoCV to find best alpha...
Computing Theta diagonal elements using LSMR...
Step 4: Computing Precision Matrix...
Using LSMR solver for large gene datasets..

/sw/pkgs/arc/python3.9-anaconda/2021.11/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:530: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.08969556912436438, tolerance: 0.07188212330071453
  model = cd_fast.enet_coordinate_descent(


Computing Theta diagonal elements using LSMR...
Step 4: Computing Precision Matrix...
Using LSMR solver for large gene datasets...
Step 5: Computing Desparsified Lasso Estimator...
                 gene      beta  std_error           pval           qval  \
36    ENSG00000025423  0.427503   0.007911   0.000000e+00   0.000000e+00   
79    ENSG00000065320  0.366333   0.009507   0.000000e+00   0.000000e+00   
170   ENSG00000088926  0.467169   0.013032  1.900712e-281  1.267141e-278   
71    ENSG00000061455  0.520170   0.015643  1.916070e-242  9.580351e-240   
67    ENSG00000055118  0.319141   0.013578  3.667310e-122  1.466924e-119   
1250  ENSG00000170091  0.208981   0.010590   1.114458e-86   3.714861e-84   
1847  ENSG00000250577  0.465700   0.028216   3.393147e-61   9.694706e-59   
162   ENSG00000086205  0.154619   0.010480   2.895864e-49   7.239660e-47   
1097  ENSG00000164161  0.322234   0.023796   8.903665e-42   1.978592e-39   
804   ENSG00000142233  0.446546   0.033485   1.433536e-40  

Computing Theta diagonal elements using LSMR...
Step 4: Computing Precision Matrix...
Using LSMR solver for large gene datasets...
Step 5: Computing Desparsified Lasso Estimator...
Empty DataFrame
Columns: [gene, beta, std_error, pval, qval, ci.lower, ci.upper]
Index: []
⚠️ Warning: No significant results for Clostridium_ramosum!
Results for Coprococcus_comes:
Reduced Feature Count: 2000
Training LassoCV to find best alpha...
Computing Theta diagonal elements using LSMR...
Step 4: Computing Precision Matrix...
Using LSMR solver for large gene datasets...
Step 5: Computing Desparsified Lasso Estimator...
                 gene     beta  std_error           pval           qval  \
1631  ENSG00000248698  0.40198   0.014247  3.897695e-175  7.795389e-172   

      ci.lower  ci.upper  
1631  0.374055  0.429905  
Results for Dorea_formicigenerans:
Reduced Feature Count: 2000
Training LassoCV to find best alpha...
Computing Theta diagonal elements using LSMR...
Step 4: Computing Precision Matrix

Computing Theta diagonal elements using LSMR...
Step 4: Computing Precision Matrix...
Using LSMR solver for large gene datasets...
Step 5: Computing Desparsified Lasso Estimator...
                 gene      beta  std_error           pval           qval  \
1689  ENSG00000241832  0.515943   0.017385  1.530277e-193  3.060555e-190   
1403  ENSG00000188107  0.651130   0.024430  1.660948e-156  1.660948e-153   
88    ENSG00000070404 -0.389972   0.018372  5.406504e-100   3.604336e-97   
1401  ENSG00000187840 -0.118873   0.013931   1.425977e-17   7.129883e-15   
1761  ENSG00000251538  0.105374   0.026422   6.659076e-05   2.663630e-02   

      ci.lower  ci.upper  
1689  0.481868  0.550019  
1403  0.603247  0.699013  
88   -0.425980 -0.353963  
1401 -0.146178 -0.091569  
1761  0.053588  0.157161  
Results for Eubacterium_siraeum:
Reduced Feature Count: 2000
Training LassoCV to find best alpha...
Computing Theta diagonal elements using LSMR...
Step 4: Computing Precision Matrix...
Using LSMR sol

Computing Theta diagonal elements using LSMR...
Step 4: Computing Precision Matrix...
Using LSMR solver for large gene datasets...
Step 5: Computing Desparsified Lasso Estimator...
Empty DataFrame
Columns: [gene, beta, std_error, pval, qval, ci.lower, ci.upper]
Index: []
⚠️ Warning: No significant results for Lachnospiraceae_bacterium_3_1_46FAA!
Results for Lachnospiraceae_bacterium_5_1_63FAA:
Reduced Feature Count: 2000
Training LassoCV to find best alpha...
Computing Theta diagonal elements using LSMR...
Step 4: Computing Precision Matrix...
Using LSMR solver for large gene datasets...
Step 5: Computing Desparsified Lasso Estimator...
                 gene      beta  std_error           pval           qval  \
669   ENSG00000138449 -0.980443   0.014930   0.000000e+00   0.000000e+00   
1830  ENSG00000258653  0.726038   0.029849  1.109218e-130  1.109218e-127   
1847  ENSG00000259278  0.385198   0.025479   1.230495e-51   8.203300e-49   
679   ENSG00000139155  0.266168   0.027056   7.7619

Computing Theta diagonal elements using LSMR...
Step 4: Computing Precision Matrix...
Using LSMR solver for large gene datasets...
Step 5: Computing Desparsified Lasso Estimator...
Empty DataFrame
Columns: [gene, beta, std_error, pval, qval, ci.lower, ci.upper]
Index: []
⚠️ Warning: No significant results for Oscillibacter_unclassified!
Results for Paraprevotella_unclassified:
Reduced Feature Count: 2000
Training LassoCV to find best alpha...
Computing Theta diagonal elements using LSMR...
Step 4: Computing Precision Matrix...
Using LSMR solver for large gene datasets...
Step 5: Computing Desparsified Lasso Estimator...
                 gene      beta  std_error           pval           qval  \
1643  ENSG00000229484  0.762558   0.015039   0.000000e+00   0.000000e+00   
1935  ENSG00000268182  0.837747   0.023425  4.172327e-280  4.172327e-277   
1994  ENSG00000273025  0.504862   0.014427  2.655346e-268  1.770231e-265   
1875  ENSG00000261025  0.240109   0.007572  1.070836e-220  5.354181e

Computing Theta diagonal elements using LSMR...
Step 4: Computing Precision Matrix...
Using LSMR solver for large gene datasets...
Step 5: Computing Desparsified Lasso Estimator...
Empty DataFrame
Columns: [gene, beta, std_error, pval, qval, ci.lower, ci.upper]
Index: []
⚠️ Warning: No significant results for Ruminococcus_obeum!
Results for Solobacterium_moorei:
Reduced Feature Count: 2000
Training LassoCV to find best alpha...
Computing Theta diagonal elements using LSMR...
Step 4: Computing Precision Matrix...
Using LSMR solver for large gene datasets...
Step 5: Computing Desparsified Lasso Estimator...
Empty DataFrame
Columns: [gene, beta, std_error, pval, qval, ci.lower, ci.upper]
Index: []
⚠️ Warning: No significant results for Solobacterium_moorei!
Results for Streptococcus_anginosus:
Reduced Feature Count: 2000
Training LassoCV to find best alpha...
Computing Theta diagonal elements using LSMR...
Step 4: Computing Precision Matrix...
Using LSMR solver for large gene datasets...


Computing Theta diagonal elements using LSMR...
Step 4: Computing Precision Matrix...
Using LSMR solver for large gene datasets...
Step 5: Computing Desparsified Lasso Estimator...
Empty DataFrame
Columns: [gene, beta, std_error, pval, qval, ci.lower, ci.upper]
Index: []
⚠️ Warning: No significant results for Streptococcus_salivarius!
Results for Streptococcus_sanguinis:
Reduced Feature Count: 2000
Training LassoCV to find best alpha...
Computing Theta diagonal elements using LSMR...
Step 4: Computing Precision Matrix...
Using LSMR solver for large gene datasets...
Step 5: Computing Desparsified Lasso Estimator...
Empty DataFrame
Columns: [gene, beta, std_error, pval, qval, ci.lower, ci.upper]
Index: []
⚠️ Warning: No significant results for Streptococcus_sanguinis!
Results for Subdoligranulum_sp_4_3_54A2FAA:
Reduced Feature Count: 2000
Training LassoCV to find best alpha...
Computing Theta diagonal elements using LSMR...
Step 4: Computing Precision Matrix...
Using LSMR solver for larg

In [8]:
final_results_ileum = pd.concat(list_ileum, ignore_index=True)
final_results_ileum.to_csv("results_ileum_all_microbes.csv", index=False)  # 保存所有微生物的结果

In [9]:
df_x_colon = pd.read_csv("x.left.colon0317.csv", delimiter=",")
df_x_colon = df_x_colon.drop(columns=["Unnamed: 0"])
X_data_colon = df_x_colon  # 保持 DataFrame 以保留基因名

df_y_colon = pd.read_csv("y.left.colon0317.csv", delimiter=",")
df_y_colon = df_y_colon.drop(columns=["Unnamed: 0"])
y_data_colon = df_y_colon.to_numpy()  # 转换成 NumPy 数组

In [10]:
list_colon = []
for j, microbe in enumerate(df_y_colon.columns):
    y_j = y_data_colon[:, j]  # 选择一个微生物作为响应变量
    print(f"Results for {microbe}:")
    results_colon = desparsified_lasso(X_data_colon, y_j, feature_selection='kbest', k_features=2000)
    print(results_colon)
    if not results_colon.empty:
        results_colon["microbe"] = microbe
        list_colon.append(results_colon)
    else:
        print(f"⚠️ Warning: No significant results for {microbe}!")

Results for Actinomyces_graevenitzii:
Reduced Feature Count: 2000
Training LassoCV to find best alpha...
Computing Theta diagonal elements using LSMR...
Step 4: Computing Precision Matrix...
Using LSMR solver for large gene datasets...
Step 5: Computing Desparsified Lasso Estimator...
Empty DataFrame
Columns: [gene, beta, std_error, pval, qval, ci.lower, ci.upper]
Index: []
⚠️ Warning: No significant results for Actinomyces_graevenitzii!
Results for Actinomyces_massiliensis:
Reduced Feature Count: 2000
Training LassoCV to find best alpha...
Computing Theta diagonal elements using LSMR...
Step 4: Computing Precision Matrix...
Using LSMR solver for large gene datasets...
Step 5: Computing Desparsified Lasso Estimator...
Empty DataFrame
Columns: [gene, beta, std_error, pval, qval, ci.lower, ci.upper]
Index: []
⚠️ Warning: No significant results for Actinomyces_massiliensis!
Results for Actinomyces_oris:
Reduced Feature Count: 2000
Training LassoCV to find best alpha...
Computing Theta dia

Computing Theta diagonal elements using LSMR...
Step 4: Computing Precision Matrix...
Using LSMR solver for large gene datasets...
Step 5: Computing Desparsified Lasso Estimator...
                 gene      beta  std_error           pval           qval  \
24    ENSG00000010379 -0.747136   0.033538  6.158299e-110  1.231660e-106   
1724  ENSG00000198488 -0.426039   0.022465   3.355537e-80   3.355537e-77   
528   ENSG00000117036 -0.359163   0.020215   1.265120e-70   8.434134e-68   
1315  ENSG00000169213 -0.446676   0.028034   3.715556e-57   1.857778e-54   
503   ENSG00000116014 -0.625823   0.042373   2.308055e-49   9.232220e-47   
595   ENSG00000124334  0.749466   0.051435   4.294626e-48   1.431542e-45   
1661  ENSG00000196368  0.444446   0.034626   1.033745e-37   2.953557e-35   
232   ENSG00000099337 -0.223418   0.018365   4.765507e-34   1.191377e-31   
1731  ENSG00000198670 -0.442847   0.044123   1.052693e-23   2.339317e-21   
352   ENSG00000105668 -0.495403   0.054038   4.831978e-20  

Computing Theta diagonal elements using LSMR...
Step 4: Computing Precision Matrix...
Using LSMR solver for large gene datasets...
Step 5: Computing Desparsified Lasso Estimator...
Empty DataFrame
Columns: [gene, beta, std_error, pval, qval, ci.lower, ci.upper]
Index: []
⚠️ Warning: No significant results for Clostridium_ramosum!
Results for Coprococcus_comes:
Reduced Feature Count: 2000
Training LassoCV to find best alpha...
Computing Theta diagonal elements using LSMR...
Step 4: Computing Precision Matrix...
Using LSMR solver for large gene datasets...
Step 5: Computing Desparsified Lasso Estimator...
Empty DataFrame
Columns: [gene, beta, std_error, pval, qval, ci.lower, ci.upper]
Index: []
⚠️ Warning: No significant results for Coprococcus_comes!
Results for Dorea_formicigenerans:
Reduced Feature Count: 2000
Training LassoCV to find best alpha...
Computing Theta diagonal elements using LSMR...
Step 4: Computing Precision Matrix...
Using LSMR solver for large gene datasets...
Step 5:

/sw/pkgs/arc/python3.9-anaconda/2021.11/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:530: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.09756855362088146, tolerance: 0.07325125836132913
  model = cd_fast.enet_coordinate_descent(


Computing Theta diagonal elements using LSMR...
Step 4: Computing Precision Matrix...
Using LSMR solver for large gene datasets...
Step 5: Computing Desparsified Lasso Estimator...
                 gene      beta  std_error           pval           qval  \
1063  ENSG00000160202  0.667201   0.025261  9.857274e-154  1.971455e-150   
1804  ENSG00000230015  0.696043   0.032354  1.175371e-102   1.175371e-99   
1120  ENSG00000163285  0.756290   0.040328   1.810705e-78   1.207137e-75   
1674  ENSG00000197632  0.481840   0.035283   1.849264e-42   9.246321e-40   
745   ENSG00000136918  0.426354   0.032779   1.117047e-38   4.468190e-36   
873   ENSG00000143882  0.363243   0.033681   4.064756e-27   1.354919e-24   
1438  ENSG00000178055  0.309581   0.032109   5.333281e-22   1.523795e-19   
1221  ENSG00000167359  0.337012   0.038888   4.465382e-18   1.116346e-15   
764   ENSG00000138083  0.280123   0.034181   2.499794e-16   5.555097e-14   
1219  ENSG00000167281  0.255083   0.036955   5.109564e-12  

...

Computing Theta diagonal elements using LSMR...
Step 4: Computing Precision Matrix...
Using LSMR solver for large gene datasets...
Step 5: Computing Desparsified Lasso Estimator...
                 gene      beta  std_error           pval           qval  \
262   ENSG00000102781  0.564447   0.017995  5.803692e-216  1.160738e-212   
1465  ENSG00000186051 -1.119929   0.036815  2.925155e-203  2.925155e-200   
496   ENSG00000122223  0.389638   0.016548  1.358679e-122  9.057861e-120   
1841  ENSG00000253147 -1.059930   0.048326  1.264377e-106  6.321883e-104   
622   ENSG00000133019 -0.520082   0.028191   5.339934e-76   2.135974e-73   
65    ENSG00000047936 -0.567579   0.032830   5.724827e-67   1.908276e-64   
981   ENSG00000157388 -0.415443   0.028055   1.301961e-49   3.719889e-47   
1771  ENSG00000236761 -0.527104   0.041501   5.846044e-37   1.461511e-34   
1106  ENSG00000164122 -0.351436   0.027769   1.043169e-36   2.318153e-34   
1496  ENSG00000187775 -0.450565   0.038320   6.428131e-32  

Computing Theta diagonal elements using LSMR...
Step 4: Computing Precision Matrix...
Using LSMR solver for large gene datasets...
Step 5: Computing Desparsified Lasso Estimator...
Empty DataFrame
Columns: [gene, beta, std_error, pval, qval, ci.lower, ci.upper]
Index: []
⚠️ Warning: No significant results for Roseburia_hominis!
Results for Roseburia_inulinivorans:
Reduced Feature Count: 2000
Training LassoCV to find best alpha...
Computing Theta diagonal elements using LSMR...
Step 4: Computing Precision Matrix...
Using LSMR solver for large gene datasets...
Step 5: Computing Desparsified Lasso Estimator...
Empty DataFrame
Columns: [gene, beta, std_error, pval, qval, ci.lower, ci.upper]
Index: []
⚠️ Warning: No significant results for Roseburia_inulinivorans!
Results for Rothia_dentocariosa:
Reduced Feature Count: 2000
Training LassoCV to find best alpha...
Computing Theta diagonal elements using LSMR...
Step 4: Computing Precision Matrix...
Using LSMR solver for large gene datasets...

Computing Theta diagonal elements using LSMR...
Step 4: Computing Precision Matrix...
Using LSMR solver for large gene datasets...
Step 5: Computing Desparsified Lasso Estimator...
                 gene      beta  std_error           pval           qval  \
212   ENSG00000100721 -0.912953   0.011781   0.000000e+00   0.000000e+00   
1694  ENSG00000228570 -0.799564   0.023461  1.457000e-254  1.457000e-251   
675   ENSG00000136573 -0.232065   0.009985  1.749335e-119  1.166223e-116   
560   ENSG00000128346 -0.474140   0.022597   9.577706e-98   4.788853e-95   
1433  ENSG00000186509 -0.611547   0.034636   9.072010e-70   3.628804e-67   
280   ENSG00000105501 -0.259708   0.017403   2.315022e-50   7.716740e-48   
408   ENSG00000115423 -0.335146   0.025852   1.962060e-38   5.605887e-36   
858   ENSG00000151577 -0.210597   0.016468   1.905183e-37   4.762956e-35   
235   ENSG00000102174 -0.293530   0.026708   4.251139e-28   9.446975e-26   
1764  ENSG00000239828 -0.191852   0.018030   1.924961e-26  

/sw/pkgs/arc/python3.9-anaconda/2021.11/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:530: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.06706310507648539, tolerance: 0.04359958430541458
  model = cd_fast.enet_coordinate_descent(


Computing Theta diagonal elements using LSMR...
Step 4: Computing Precision Matrix...
Using LSMR solver for large gene datasets...
Step 5: Computing Desparsified Lasso Estimator...
                 gene      beta  std_error          pval          qval  \
1987  ENSG00000272301 -0.506410   0.026111  8.599106e-84  1.719821e-80   
1613  ENSG00000187922 -0.241986   0.016475  7.648018e-49  7.648018e-46   
542   ENSG00000117318 -0.326196   0.025961  3.296564e-36  2.197710e-33   
189   ENSG00000087842 -0.222258   0.018894  6.019427e-32  3.009713e-29   
1428  ENSG00000174502 -0.358097   0.031034  8.400654e-31  3.360262e-28   
1562  ENSG00000184261 -0.302432   0.028555  3.276873e-26  1.092291e-23   
1979  ENSG00000271998 -0.315289   0.030909  1.970509e-24  5.630026e-22   
1215  ENSG00000163794 -0.226364   0.025364  4.468621e-19  1.117155e-16   
1354  ENSG00000170054 -0.148332   0.017142  5.020396e-18  1.115644e-15   
1177  ENSG00000162771 -0.207307   0.024073  7.198600e-18  1.439720e-15   
1820 

In [11]:
final_results_colon = pd.concat(list_colon, ignore_index=True)
final_results_colon.to_csv("results_colon_all_microbes.csv", index=False)  # 保存所有微生物的结果